In [118]:
#| default_exp dmelements

# dmelements
> Operators on density-matrix elements

## Imports -

In [119]:
#| hide
from fastcore.test import test_close

In [120]:
#| export
import importlib

from pylgs.imports import *
from pylgs.utilities.nbdev import DictTbl, AttributeTbl
from pylgs.utilities.testing import test_array
from pylgs.utilities.sparse import sparse_kronecker_matrix, sparse_toeplitz, sparse_identity, sparse_diag, sparse
from pylgs.utilities.numpy import sym_range
from pylgs.pymor.parameters import *
from pylgs.pymor.vectorarrays import *
from pylgs.pymor.operators import *
from pylgs.pymor.timestepping import *
from pylgs.pymor.grids import *
from pylgs.pymor.models import *
from pymor.vectorarrays.interface import VectorArray
from pymor.operators.interface import Operator

In [121]:
#| hide
np.set_printoptions(formatter={'float': lambda x: f'{x:^ 8.2}' if x else f'{0:^ 8}'}, linewidth=140)

## DM elements -

In [122]:
with importlib.resources.path("pylgs.systems.NaD1_Toy", "Flux.mtxn") as path:
    flux = LincombOperator.from_file(path)

In [123]:
flux.assemble().source.dims

('Density matrix',)

In [124]:
flux.assemble().matrix

Format,coo
Data Type,float64
Shape,"(1, 4)"
nnz,1
Density,0.25
Read-only,True
Size,24
Storage ratio,0.75


In [125]:
#| hide
with importlib.resources.path("pylgs.systems.NaD1_Toy", "Flux.mtxn") as path:
    flux = LincombOperator.from_file(path)
dm_elements = flux.assemble().matrix['Density matrix' + Lbl.SRC].rename({"Density matrix (source)": "Density matrix"})
dm_element = dm_elements[0].item()
dm_element

'ρ<sub>Re, 3S<sub>1/2</sub>, 3S<sub>1/2</sub></sub>'

In [126]:
#| export
_dm_element_hfz = re.compile(r'ρ<sub>([^,]+), \(([^,]+), ([^,]+), ([^)]+)\), \(([^,]+), ([^,]+), ([^)]+)\)</sub>')
_dm_element_toy = re.compile(r'ρ<sub>([^,]+), ([^,]+), ([^,]+)</sub>')

### parse_dm_element -


In [127]:
#| export
def parse_dm_element(s):
    if match := _dm_element_hfz.findall(s):
        return match[0][0], match[0][1:4], match[0][4:7] 
    if match := _dm_element_toy.findall(s): 
        return match[0][0], (match[0][1],), (match[0][2],)

In [128]:
#| hide
[parse_dm_element(e) for e in dm_elements.data]

[('Re', ('3S<sub>1/2</sub>',), ('3S<sub>1/2</sub>',)),
 ('Re', ('3S<sub>1/2</sub>',), ('3P<sub>1/2</sub>',)),
 ('Im', ('3S<sub>1/2</sub>',), ('3P<sub>1/2</sub>',)),
 ('Re', ('3P<sub>1/2</sub>',), ('3P<sub>1/2</sub>',))]

### population_element -


In [129]:
#| export
def population_element(s):
    parsed_dm = parse_dm_element(s)
    return parsed_dm[1] == parsed_dm[2]

In [130]:
#| hide
[population_element(e) for e in dm_elements.data]

[True, False, False, True]

### level_label -


In [131]:
#| export
@dispatch
def level_label(expr:str):
    parsed = parse_dm_element(expr)
    if parsed[1][0] != parsed[2][0]: return ''
    return parsed[1][0]

@dispatch
def level_label(expr:DataArray):
    return xr.apply_ufunc(level_label, expr, vectorize=True)

In [132]:
#| hide
level_label(dm_element)

'3S<sub>1/2</sub>'

In [133]:
#| hide
level_label(dm_elements)

<xarray.DataArray 'Density matrix (source)' (Density matrix: 4)> Size: 256B
array(['3S<sub>1/2</sub>', '', '', '3P<sub>1/2</sub>'], dtype='<U16')
Coordinates:
  * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...

### levels -

In [134]:
#| export
def levels(dm_elements):
    return xr.Variable(
        'Level', 
        list(set().union(filter(None, xr.apply_ufunc(level_label, dm_elements, vectorize=True).data)))
    )

In [135]:
#| hide
levels(dm_elements)

<xarray.Variable (Level: 2)> Size: 128B
array(['3S<sub>1/2</sub>', '3P<sub>1/2</sub>'], dtype='<U16')

### population -

In [136]:
# #| export
# # Make this a projection operator onto populations (diagonal elements)
# # Then can have sum operators to sum over hyperfine, fine, or all elements
# def population(dm_elements):
#     return ScaleOperator(
#         xr.apply_ufunc(population_element, dm_elements, vectorize=True).astype(bool),
#         name="Population"
#     )

In [137]:
# #| export
# def population(dm_elements):
#     mask = [population_element(e) for e in dm_elements.data]
#     return XarrayMatrixOperator(
#         DataArray(
#             np.identity(len(dm_elements.data)) * mask,
#             coords=[
#                 ("Density matrix (range)", dm_elements.data),
#                 ("Density matrix (source)", dm_elements.data), 
#             ],
#         )
#     )

In [156]:
#| export
def population(dm_elements):
    elements = dm_elements.data
    mask = [population_element(e) for e in elements]
    pop_labels = [level_label(e) for e in elements[mask]]
    return XarrayMatrixOperator(
        DataArray(
            np.identity(len(elements))[mask],
            coords=[
                ("Population (range)", pop_labels),
                ("Density matrix (source)", elements), 
            ],
        )
    )

In [157]:
op = population(dm_elements)
op

XarrayMatrixOperator(
    <xarray.DataArray (Population (range): 2, Density matrix (source): 4)> Size: 64B
    array([[   1.0  ,     0   ,     0   ,     0   ],
           [    0   ,     0   ,     0   ,    1.0  ]])
    Coordinates:
      * Population (range)       (Population (range)) <U16 128B '3S<sub>1/2</sub>...
      * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...,
    source=XarrayVectorSpace(
               coords=Coordinates:
                        * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...),
    range=XarrayVectorSpace(
              coords=Coordinates:
                       * Population  (Population) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>'))

In [158]:
op.source

XarrayVectorSpace(
    coords=Coordinates:
             * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...)

In [159]:
op.range

XarrayVectorSpace(
    coords=Coordinates:
             * Population  (Population) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>')

In [160]:
op.matrix

<xarray.DataArray (Population (range): 2, Density matrix (source): 4)> Size: 64B
array([[   1.0  ,     0   ,     0   ,     0   ],
       [    0   ,     0   ,     0   ,    1.0  ]])
Coordinates:
  * Population (range)       (Population (range)) <U16 128B '3S<sub>1/2</sub>...
  * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...

In [161]:
dm = op.source.ones()

In [162]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(
        coords=Coordinates:
                   Density matrix  (Population) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>'),
    XarrayVectorArrayImpl(
        <xarray.DataArray (Population: 2)> Size: 16B
        array([   1.0  ,    1.0  ])
        Coordinates:
          * Population  (Population) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>',
        <xarray.DataArray 'XarrayVectorSpace' (Population: 2)> Size: 16B
        array([    0   ,     0   ])
        Coordinates:
            Density matrix  (Population) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>'
        Dimensions without coordinates: Population),
    _len=1)

In [163]:
pop.array

<xarray.DataArray (Population: 2)> Size: 16B
array([   1.0  ,    1.0  ])
Coordinates:
  * Population  (Population) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>'

### total_population - 

In [164]:
op.matrix.sum("Population (range)")

<xarray.DataArray (Density matrix (source): 4)> Size: 32B
array([   1.0  ,     0   ,     0   ,    1.0  ])
Coordinates:
  * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...

In [175]:
#| export
def total_population(dm_elements):
    return XarrayMatrixOperator(population(dm_elements).matrix.sum("Population (range)"), name="Total population")

In [176]:
op = total_population(dm_elements)

In [177]:
op.source

XarrayVectorSpace(
    coords=Coordinates:
             * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...)

In [178]:
op.range

XarrayVectorSpace(
    coords=Coordinates:
               *empty*)

In [179]:
op.matrix

<xarray.DataArray (Density matrix (source): 4)> Size: 32B
array([   1.0  ,     0   ,     0   ,    1.0  ])
Coordinates:
  * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...

In [180]:
dm = op.source.ones()

In [181]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(
        coords=Coordinates:
                   *empty*,
        name='Total population'),
    XarrayVectorArrayImpl(
        <xarray.DataArray 'Total population' ()> Size: 8B
        array(   2.0  ),
        <xarray.DataArray 'Total population' ()> Size: 8B
        array(    0   )),
    _len=1)

In [182]:
pop.array

<xarray.DataArray 'Total population' ()> Size: 8B
array(   2.0  )

### level_population -


In [183]:
#|export
def level_population(dm_elements):
    return XarrayMatrixOperator(
            xr.concat([level == level_label(dm_elements) for level in levels(dm_elements)], levels(dm_elements)), 
            source="Density matrix",
            range="Level",
            name="Population"
        )    

In [184]:
op = level_population(dm_elements)
op

XarrayMatrixOperator(
    <xarray.DataArray 'Density matrix (source)' (Level (range): 2,
                                                 Density matrix (source): 4)> Size: 8B
    array([[ True, False, False, False],
           [False, False, False,  True]])
    Coordinates:
      * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...
      * Level (range)            (Level (range)) <U16 128B '3S<sub>1/2</sub>' '3P...,
    source=XarrayVectorSpace(
               coords=Coordinates:
                        * Density matrix  (Density matrix) <U50 800B 'ρ<sub>Re, 3S<sub>1/2</sub>, 3...),
    range=XarrayVectorSpace(
              coords=Coordinates:
                       * Level    (Level) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>'),
    name='Population')

In [185]:
op.matrix

<xarray.DataArray 'Density matrix (source)' (Level (range): 2,
                                             Density matrix (source): 4)> Size: 8B
array([[ True, False, False, False],
       [False, False, False,  True]])
Coordinates:
  * Density matrix (source)  (Density matrix (source)) <U50 800B 'ρ<sub>Re, 3...
  * Level (range)            (Level (range)) <U16 128B '3S<sub>1/2</sub>' '3P...

In [186]:
dm = op.source.ones()

In [187]:
pop = op.apply(dm)
pop

XarrayVectorArray(
    XarrayVectorSpace(
        coords=Coordinates:
                 * Level    (Level) int64 16B 0 1,
        name='Population'),
    XarrayVectorArrayImpl(
        <xarray.DataArray 'Population' (Level: 2)> Size: 16B
        array([   1.0  ,    1.0  ])
        Coordinates:
          * Level    (Level) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>',
        <xarray.DataArray 'Population' (Level: 2)> Size: 16B
        array([    0   ,     0   ])
        Coordinates:
          * Level    (Level) int64 16B 0 1),
    _len=1)

In [188]:
pop.array

<xarray.DataArray 'Population' (Level: 2)> Size: 16B
array([   1.0  ,    1.0  ])
Coordinates:
  * Level    (Level) <U16 128B '3S<sub>1/2</sub>' '3P<sub>1/2</sub>'

### sum_over_dm -


In [189]:
#| export
def sum_over_dm(dm_elements):
    return SumOperator({'Density matrix' + Lbl.SRC: dm_elements})

## Export -

In [190]:
#| hide
import nbdev; nbdev.nbdev_export()